# PDF -> UI PDF (Enterprise Exact Copy)
### Same file, no content change, no quality loss
---
> **How to use:**
> 1. Run Cell 2 once
> 2. Run Cell 3
> 3. Click **Import PDF** (or use **Browse/Path** fallback)
> 4. Click **Create Exact Copy**
>
> The notebook saves a new PDF in your **Downloads** folder with byte-level integrity verification.

In [ ]:
# Install ipywidgets if not already installed
%pip install -q ipywidgets

In [ ]:
import base64
import hashlib
from datetime import datetime
from html import escape
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display


# mdtopdf-like colors
BG = "#fff2cc"
BORDER = "#c4a84a"
TEXT = "#2c2417"


def set_status(message, color="#8b6914"):
    status.value = (
        f"<div style='margin:8px 0; color:{color}; font-family:Arial, sans-serif;'>"
        f"{escape(message)}</div>"
    )


def get_uploaded_pdf():
    value = import_button.value
    if not value:
        return None, None

    if isinstance(value, dict):
        file_name, item = next(iter(value.items()))
        return file_name, bytes(item.get("content", b""))

    if isinstance(value, tuple):
        item = value[0]
        return item.get("name", "uploaded.pdf"), bytes(item.get("content", b""))

    return None, None


def get_pdf_from_path(raw_path):
    cleaned = raw_path.strip().strip('"').strip("'")
    if not cleaned:
        set_status("Enter a PDF path or import a file first.", color="#991b1b")
        return None, None

    p = Path(cleaned).expanduser()
    if not p.exists() or not p.is_file():
        set_status("Path not found. Please check and try again.", color="#991b1b")
        return None, None

    if p.suffix.lower() != ".pdf":
        set_status("Selected file is not a PDF.", color="#991b1b")
        return None, None

    try:
        return p.name, p.read_bytes()
    except Exception as exc:
        set_status(f"Unable to read file: {exc}", color="#991b1b")
        return None, None


def create_exact_copy(file_name, file_bytes):
    src_name = Path(file_name).name
    stem = Path(src_name).stem

    out_dir = Path.home() / "Downloads"
    out_dir.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_name = f"{stem}_ui_exact_{stamp}.pdf"
    out_path = out_dir / out_name

    # Write raw bytes unchanged
    out_path.write_bytes(file_bytes)

    output_bytes = out_path.read_bytes()
    in_md5 = hashlib.md5(file_bytes).hexdigest()
    out_md5 = hashlib.md5(output_bytes).hexdigest()
    identical = in_md5 == out_md5 and len(file_bytes) == len(output_bytes)

    set_status(
        "Exact copy created successfully." if identical else "Copy created, but integrity verification failed.",
        color="#166534" if identical else "#991b1b",
    )

    with output_area:
        clear_output()

        info_html = f"""
<div style="background:{BG}; border:2px solid {BORDER}; border-radius:12px; padding:16px; margin:10px 0; color:{TEXT}; font-family:Arial,sans-serif;">
  <h3 style="margin:0 0 10px 0;">Copy Summary</h3>
  <p style="margin:5px 0;"><b>Input PDF:</b> {escape(src_name)}</p>
  <p style="margin:5px 0;"><b>Output PDF:</b> {escape(str(out_path))}</p>
  <p style="margin:5px 0;"><b>Input MD5:</b> {in_md5}</p>
  <p style="margin:5px 0;"><b>Output MD5:</b> {out_md5}</p>
  <p style="margin:5px 0;"><b>Integrity:</b> {'PASS (1:1 identical)' if identical else 'FAIL'}</p>
</div>
        """
        display(HTML(info_html))

        # Optional browser download button for smaller PDFs
        if len(output_bytes) <= 15 * 1024 * 1024:
            b64 = base64.b64encode(output_bytes).decode("ascii")
            button_html = f"""
<div style="text-align:center; margin:20px 0;">
  <a href="data:application/pdf;base64,{b64}" download="{escape(out_name)}"
     style="display:inline-block; padding:14px 40px; background-color:{BORDER}; color:#fff;
            font-family:Arial,sans-serif; font-size:15px; font-weight:700; text-decoration:none;
            border-radius:8px; box-shadow:0 3px 10px rgba(0,0,0,0.15);">
     Download {escape(out_name)}
  </a>
</div>
            """
            display(HTML(button_html))
        else:
            display(HTML(
                f"<p style='color:{TEXT}; font-family:Arial,sans-serif;'>"
                "File is large, download button is skipped. "
                f"Use the saved file from: <b>{escape(str(out_path))}</b></p>"
            ))


def on_browse_click(_):
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askopenfilename(
            title="Select PDF file",
            filetypes=[("PDF files", "*.pdf")],
        )
        root.destroy()

        if not selected:
            set_status("No file selected.", color="#b45309")
            return

        path_input.value = selected
        set_status("Path selected. Click 'Create Exact Copy'.", color="#8b6914")
    except Exception as exc:
        set_status(f"Browse is unavailable: {exc}. Use path input.", color="#991b1b")


def on_create_click(_):
    file_name, file_bytes = get_uploaded_pdf()

    if file_bytes is None:
        file_name, file_bytes = get_pdf_from_path(path_input.value)

    if file_bytes is None:
        return

    create_exact_copy(file_name, file_bytes)


# Top UI controls
import_button = widgets.FileUpload(
    accept=".pdf",
    multiple=False,
    description="Import PDF",
    button_style="warning",
)
create_button = widgets.Button(
    description="Create Exact Copy",
    button_style="success",
    icon="check",
)

# Path fallback
path_input = widgets.Text(
    value="",
    placeholder="C:/Users/YourName/Downloads/file.pdf",
    description="Path:",
    layout=widgets.Layout(width="68%"),
)
browse_button = widgets.Button(description="Browse", button_style="info", icon="folder-open")

status = widgets.HTML()
output_area = widgets.Output()

browse_button.on_click(on_browse_click)
create_button.on_click(on_create_click)

header = widgets.HTML(
    "<h2 style='margin:0; color:#1a1206; font-family:Arial,sans-serif;'>PDF -> UI PDF (Exact Enterprise Copy)</h2>"
)
help_text = widgets.HTML(
    f"<div style='background:{BG}; border:2px solid {BORDER}; border-radius:12px; padding:12px; margin:8px 0; color:{TEXT}; font-family:Arial,sans-serif;'>"
    "1) Click <b>Import PDF</b> at top, then <b>Create Exact Copy</b>.<br>"
    "2) If upload is blocked, click <b>Browse</b> or paste full path, then click <b>Create Exact Copy</b>.<br>"
    "3) Output is saved in Downloads with MD5 integrity verification."
    "</div>"
)

ui = widgets.VBox([
    header,
    help_text,
    widgets.HBox([import_button, create_button]),
    widgets.HBox([path_input, browse_button]),
    status,
    output_area,
])

set_status("Ready. Import a PDF to create a 1:1 exact copy.", color="#1d4ed8")
display(ui)